# CMPE 460 BiLSTM Training on Mendeley

**Member 2**

This notebook trains BiLSTM on the Mendeley archive produced by `06_mendeley_data_preprocessing.ipynb`. The archive supplies the fixed splits, training vocabulary, six label names, and 512-character input length.

The architecture and training settings follow `04_bilstm_secureai.ipynb`. Training uses seed 42, 32-dimensional embeddings, 64 hidden units per direction, dropout of 0.3, batch size 64, and Adam with a learning rate of 0.001. Validation macro-F1 selects the best epoch.

The device and PyTorch version are printed with the results. Compare runtime measurements under a common hardware and software environment. The best model, preprocessing metadata, and training history are saved under `data/mendeley/`.


## 1. Load the Prepared Data

The archive records the source CSV checksum, input template, row assignments, vocabulary, and sequence length. This notebook loads its training and validation arrays. The file checksum is printed to identify the prepared input used by the experiment.

If imports fail, install dependencies once using `%pip install torch numpy pandas scikit-learn` in a separate code cell.


In [1]:
from time import perf_counter

setup_started = perf_counter()
print("Loading packages and prepared Mendeley data...", flush=True)

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_PATH = Path("data/mendeley/preprocessed.npz")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Run all cells in 06_mendeley_data_preprocessing.ipynb "
        f"to create {DATA_PATH.resolve()} in this checkout."
    )

with np.load(DATA_PATH, allow_pickle=False) as prepared:
    metadata = json.loads(prepared["metadata_json"].item())

    if (
        metadata.get("schema_version") != 1
        or metadata.get("dataset_id") != "zhzd5j9ffy"
        or metadata.get("dataset_version") != 2
    ):
        raise ValueError("The prepared file does not match the expected Mendeley preprocessing schema.")

    X = {
        split: prepared[f"X_{split}"]
        for split in ("train", "validation")
    }
    y = {
        split: prepared[f"y_{split}"]
        for split in ("train", "validation")
    }

MAX_LENGTH = metadata["max_length"]
PAD_ID = metadata["pad_id"]
UNK_ID = metadata["unk_id"]
char_to_id = metadata["char_to_id"]
label_names = metadata["label_names"]
DATASET_REVISION = metadata["dataset_doi"]

for split in ("train", "validation"):
    assert X[split].dtype == np.int64
    assert y[split].dtype == np.int64
    assert X[split].shape == (metadata["split_sizes"][split], MAX_LENGTH)
    assert y[split].shape == (metadata["split_sizes"][split],)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < metadata["vocabulary_size"]
    assert np.all((y[split] >= 0) & (y[split] < len(label_names)))
    print(f"{split}: X={X[split].shape}, y={y[split].shape}", flush=True)

prepared_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print(f"Prepared file SHA-256: {prepared_sha256}", flush=True)
print(f"Source CSV SHA-256: {metadata['source_csv_sha256']}", flush=True)
print(f"Data loaded in {perf_counter() - setup_started:.1f}s.", flush=True)
display(pd.Series(label_names, name="class").rename_axis("label_id").to_frame())


Loading packages and prepared Mendeley data...


train: X=(18015, 512), y=(18015,)


validation: X=(2573, 512), y=(2573,)


Prepared file SHA-256: 5308146e1ea6f365e4ac49dec8e3a43449a1d050f3b48abc1f44adba9477469e


Source CSV SHA-256: d3b3509555924de6e1d37522806f2debd97ea6659e8254a95ad98d19c28a24d6


Data loaded in 2.2s.


## 2. Configure Training and Create Data Loaders

The seed, architecture dimensions, dropout, batch size, optimizer settings, maximum epochs, and early-stopping patience follow the SecureAI-SE BiLSTM experiment. Only the training loader shuffles the rows.

The CPU thread count is two. When CUDA is available, the notebook uses the GPU. The actual environment and epoch durations are recorded with the experiment.


In [2]:
SEED = 42
BATCH_SIZE = 64
EMBEDDING_DIM = 32
HIDDEN_SIZE = 64
CPU_THREADS = 2
DROPOUT = 0.3
LEARNING_RATE = 0.001
MAX_EPOCHS = 15
PATIENCE = 4
LOG_EVERY_BATCHES = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(CPU_THREADS)
vocabulary_size = len(char_to_id) + 2
number_of_classes = len(label_names)

loaders = {}
for split in ("train", "validation"):
    samples = TensorDataset(
        torch.from_numpy(X[split]),
        torch.from_numpy(y[split]),
    )
    loaders[split] = DataLoader(
        samples,
        batch_size=BATCH_SIZE,
        shuffle=(split == "train"),
        generator=torch.Generator().manual_seed(SEED),
        num_workers=0,
    )

print(f"Device: {device}", flush=True)
print(f"PyTorch version: {torch.__version__}", flush=True)
print(f"CPU threads: {torch.get_num_threads()}", flush=True)
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
else:
    print("This kernel will train on the CPU.", flush=True)
print(f"Dataset revision: {DATASET_REVISION}", flush=True)
print(f"Vocabulary size: {vocabulary_size}", flush=True)
print(f"Number of classes: {number_of_classes}", flush=True)
print(f"Maximum input length: {MAX_LENGTH}", flush=True)
print(f"Hidden units per direction: {HIDDEN_SIZE}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Learning rate: {LEARNING_RATE}", flush=True)
print(f"Maximum epochs: {MAX_EPOCHS}", flush=True)
print(f"Early-stopping patience: {PATIENCE}", flush=True)


Device: cuda


PyTorch version: 2.7.0+cu128


CPU threads: 12


GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Dataset revision: 10.17632/zhzd5j9ffy.2


Vocabulary size: 222


... (3 outputs left out)

Batch size: 64


Learning rate: 0.001


Maximum epochs: 15


Early-stopping patience: 4


## 3. Define the BiLSTM

The embedding layer maps character IDs to learned vectors. One bidirectional LSTM layer processes the request in both directions. The final forward and backward hidden states are concatenated, passed through dropout, and classified into six categories.

Packing uses the true non-padding sequence lengths, which keeps padding outside the recurrent computation.

[PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) · [Packed sequences](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html)


In [3]:
class BiLSTM(nn.Module):
    def __init__(
        self,
        vocabulary_size,
        number_of_classes,
        embedding_dim=32,
        hidden_size=64,
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=pad_id,
        )
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, number_of_classes)

    def forward(self, tokens):
        lengths = tokens.ne(self.pad_id).sum(dim=1).cpu()
        embedded = self.embedding(tokens)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden_states, _) = self.lstm(packed)
        combined = torch.cat(
            [hidden_states[-2], hidden_states[-1]],
            dim=1,
        )
        return self.classifier(self.dropout(combined))


model = BiLSTM(
    vocabulary_size=vocabulary_size,
    number_of_classes=number_of_classes,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    pad_id=PAD_ID,
).to(device)

for split in ("train", "validation"):
    assert (X[split] != PAD_ID).sum(axis=1).min() > 0

model.eval()
with torch.no_grad():
    example_inputs = torch.from_numpy(X["train"][:BATCH_SIZE]).to(device)
    example_scores = model(example_inputs)

assert example_scores.shape == (len(example_inputs), number_of_classes)
assert torch.isfinite(example_scores).all()

print(model)
print(f"Example output shape: {tuple(example_scores.shape)}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


BiLSTM(
  (embedding): Embedding(222, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=6, bias=True)
)
Example output shape: (64, 6)
Trainable parameters: 58,054


## 4. Train and Select the Best Epoch

Cross-entropy compares the class scores with the integer labels. Training progress is printed after the first batch, every 20 batches, and the final batch of each epoch. The first batch may take longer while PyTorch initializes its execution kernels.

After each epoch, validation accuracy and macro-F1 are calculated. The best weights are kept in memory. Training stops after four consecutive epochs without an improvement in validation macro-F1. Epoch durations are recorded alongside the metrics.


In [4]:
from copy import deepcopy

from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    true_labels = []
    predicted_labels = []

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        scores = model(inputs)
        loss = criterion(scores, targets)

        total_loss += loss.item() * targets.size(0)
        true_labels.extend(targets.cpu().tolist())
        predicted_labels.extend(scores.argmax(dim=1).cpu().tolist())

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            labels=list(range(number_of_classes)),
            average="macro",
            zero_division=0,
        ),
    }
    return metrics, true_labels, predicted_labels


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history_rows = []

print(f"Starting training for up to {MAX_EPOCHS} epochs on {device}.", flush=True)

training_started = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = perf_counter()
    model.train()
    total_training_loss = 0.0

    for batch_index, (inputs, targets) in enumerate(loaders["train"], start=1):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(inputs)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        total_training_loss += loss.item() * targets.size(0)

        if (
            batch_index == 1
            or batch_index % LOG_EVERY_BATCHES == 0
            or batch_index == len(loaders["train"])
        ):
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
                f"batch {batch_index:02d}/{len(loaders['train'])} | "
                f"elapsed {perf_counter() - epoch_started:.1f}s",
                flush=True,
            )

    training_loss = total_training_loss / len(loaders["train"].dataset)
    validation, _, _ = evaluate_model(model, loaders["validation"])
    epoch_seconds = perf_counter() - epoch_started

    history_rows.append({
        "epoch": epoch,
        "train_loss": training_loss,
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["accuracy"],
        "validation_macro_f1": validation["macro_f1"],
        "epoch_seconds": epoch_seconds,
    })

    print(
        f"Epoch {epoch:02d} | train loss {training_loss:.4f} | "
        f"val loss {validation['loss']:.4f} | "
        f"val accuracy {validation['accuracy']:.4f} | "
        f"val macro-F1 {validation['macro_f1']:.4f} | "
        f"time {epoch_seconds:.1f}s",
        flush=True,
    )

    if validation["macro_f1"] > best_macro_f1:
        best_macro_f1 = validation["macro_f1"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after epoch {epoch}.")
        break

model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(history_rows)

print(f"Restored the best epoch: {best_epoch}")
print(f"Total training time: {perf_counter() - training_started:.1f}s")
display(history.round(4))


Starting training for up to 15 epochs on cuda.


Epoch 01/15 | batch 01/282 | elapsed 0.3s


Epoch 01/15 | batch 20/282 | elapsed 1.7s


Epoch 01/15 | batch 40/282 | elapsed 3.2s


Epoch 01/15 | batch 60/282 | elapsed 4.8s


Epoch 01/15 | batch 80/282 | elapsed 6.3s


... (248 outputs left out)

Epoch 15/15 | batch 282/282 | elapsed 22.6s


Epoch 15 | train loss 0.1351 | val loss 0.1225 | val accuracy 0.9662 | val macro-F1 0.9642 | time 23.8s


Restored the best epoch: 15
Total training time: 348.9s


,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1,epoch_seconds
0,1,1.2821,1.0648,0.5985,0.3075,22.3177
1,2,0.9128,0.7082,0.7501,0.6060,22.3708
2,3,0.7017,0.5559,0.8092,0.7080,23.8922
3,4,0.4986,0.5512,0.8142,0.7425,23.3604
4,5,0.3848,0.3248,0.9110,0.8902,23.1311
5,6,0.3694,0.2655,0.9281,0.9158,23.1973
6,7,0.2660,0.2582,0.9254,0.9133,23.7689
7,8,0.2291,0.1999,0.9468,0.9409,22.7025
8,9,0.2217,0.1787,0.9491,0.9442,23.0159
9,10,0.1805,0.1812,0.9429,0.9362,23.6960


## 5. Report Validation Results

The following metrics describe the model selected using validation macro-F1. Record them as development results in the architecture comparison. Test metrics will be reported after the experiment settings have been selected.

Save this notebook with its outputs after the run completes. Run all cells again to start a new experiment from the recorded seed.


In [5]:
from sklearn.metrics import classification_report

validation_metrics, validation_labels, validation_predictions = evaluate_model(
    model,
    loaders["validation"],
)

print(f"Best epoch: {best_epoch}")
print(f"Validation accuracy: {validation_metrics['accuracy']:.4f}")
print(f"Validation macro-F1: {validation_metrics['macro_f1']:.4f}")
print()
print(classification_report(
    validation_labels,
    validation_predictions,
    labels=list(range(number_of_classes)),
    target_names=label_names,
    digits=4,
    zero_division=0,
))


Best epoch: 15
Validation accuracy: 0.9662
Validation macro-F1: 0.9642

              precision    recall  f1-score   support

      normal     0.9701    0.9733    0.9717      1200
         xss     0.9672    0.9672    0.9672       305
        sqli     0.9938    0.9907    0.9922       321
        cmdi     0.9618    0.9805    0.9711       257
         lfi     0.9082    0.8900    0.8990       300
        ssrf     0.9894    0.9789    0.9841       190

    accuracy                         0.9662      2573
   macro avg     0.9651    0.9634    0.9642      2573
weighted avg     0.9661    0.9662    0.9661      2573



## 6. Save and Verify the Best Model

The checkpoint contains the selected weights, architecture settings, preprocessing configuration, validation metrics, and training history. Tensors are stored on the CPU so they can be loaded on another supported device. A temporary file is replaced atomically after saving.

The cell reloads the saved tensors and checks that the restored model reproduces a validation probe. The generated checkpoint is stored in the ignored `data/` directory for later test evaluation.

[PyTorch model saving and loading](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)


In [6]:
import os
from tempfile import NamedTemporaryFile

CHECKPOINT_PATH = Path("data/mendeley/bilstm_best.pt")

model_config = {
    "vocabulary_size": vocabulary_size,
    "number_of_classes": number_of_classes,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "pad_id": PAD_ID,
}
training_config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
}

checkpoint = {
    "schema_version": 1,
    "model_name": "BiLSTM",
    "model_config": model_config,
    "training_config": training_config,
    "model_state_dict": {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    },
    "preprocessing_metadata": metadata,
    "prepared_file_sha256": prepared_sha256,
    "best_epoch": int(best_epoch),
    "epochs_run": len(history_rows),
    "training_completed": True,
    "validation_metrics": {
        name: float(value)
        for name, value in validation_metrics.items()
    },
    "history": [
        {
            name: int(value) if name == "epoch" else float(value)
            for name, value in row.items()
        }
        for row in history_rows
    ],
    "environment": {
        "pytorch": str(torch.__version__),
        "device": str(device),
        "cpu_threads": torch.get_num_threads(),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    },
}

CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_path = None
try:
    with NamedTemporaryFile(
        dir=CHECKPOINT_PATH.parent,
        prefix="bilstm-best-",
        suffix=".pt",
        delete=False,
    ) as output_file:
        temporary_path = Path(output_file.name)
        torch.save(checkpoint, output_file)
        output_file.flush()
        os.fsync(output_file.fileno())
    os.replace(temporary_path, CHECKPOINT_PATH)
finally:
    if temporary_path is not None:
        temporary_path.unlink(missing_ok=True)

restored = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)
assert restored["model_config"] == model_config
assert restored["prepared_file_sha256"] == prepared_sha256

for name, tensor in checkpoint["model_state_dict"].items():
    torch.testing.assert_close(
        restored["model_state_dict"][name],
        tensor,
        rtol=0,
        atol=0,
    )

restored_model = BiLSTM(**restored["model_config"]).to(device)
restored_model.load_state_dict(restored["model_state_dict"])
restored_model.eval()

with torch.no_grad():
    probe_inputs = torch.from_numpy(X["validation"][:8]).to(device)
    torch.testing.assert_close(
        model(probe_inputs),
        restored_model(probe_inputs),
        rtol=1e-5,
        atol=1e-5,
    )

print(f"Best model saved to {CHECKPOINT_PATH.as_posix()}")
print(f"Checkpoint size: {CHECKPOINT_PATH.stat().st_size / 1024:.1f} KiB")
print("Reloaded weights and predictions verified successfully.")


Best model saved to data/mendeley/bilstm_best.pt
Checkpoint size: 236.2 KiB
Reloaded weights and predictions verified successfully.
